In [ ]:
import os
import re
from pathlib import Path

import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()

# check if MPL_STYLE is set in the environment, else use default
if "MPL_STYLE" not in os.environ:
    os.environ["MPL_STYLE"] = "seaborn-v0_8-notebook"
plt.style.use(os.environ["MPL_STYLE"])

In [ ]:
SIM_ATTR_PATTERN = re.compile(r"Rad(\d{2})-zmet(\d{4})-M(\d)-(\d{4})")

OUTPUT_BASE = Path(os.getenv("OUTPUT_BASE"))
raw_joblib_root_path = (OUTPUT_BASE / "cache" / "raw").resolve(strict=True)
record_df = pd.DataFrame(
    {
        "init_gc_radius": int(match.group(1)),
        "init_metallicity": int(match.group(2)),
        "init_mass_lv": int(match.group(3)),
        "init_pos": int(match.group(4)),
        "filename": joblib_path.name,
    }
    for joblib_path in raw_joblib_root_path.glob("*.joblib")
    if (match := SIM_ATTR_PATTERN.match(joblib_path.stem))
)


In [ ]:
INIT_MASS_LV_TO_PLOT = [1, 5, 8]
SAMPLE_RANDOM_STATE = 114514


def _load_joblib(joblib_path: Path) -> np.ndarray:
    return joblib.load(Path(joblib_path).resolve(strict=True))[0]["stars"][
        ["x", "y"]
    ].to_numpy()


group_keys = [
    "init_gc_radius",
    "init_metallicity",
    "init_mass_lv",
]
eligible_record_df = record_df.loc[
    record_df["init_mass_lv"].isin(INIT_MASS_LV_TO_PLOT)
]
sample_order = np.random.default_rng(SAMPLE_RANDOM_STATE).permutation(
    len(record_df.loc[
    record_df["init_mass_lv"].isin(INIT_MASS_LV_TO_PLOT)
])
)
sample_record_df = (
    eligible_record_df.assign(sample_order=sample_order)
    .sort_values(["sample_order", *group_keys], kind="stable")
    .drop_duplicates(group_keys)
    .sort_values(group_keys, kind="stable")
    .loc[:, group_keys + ["filename"]]
    .reset_index(drop=True)
)
sample_records = list(sample_record_df.itertuples(index=False, name=None))
sample_data_dict = {
    (init_gc_radius, init_metallicity, init_mass_lv): _load_joblib(
        raw_joblib_root_path / filename
    )
    for init_gc_radius, init_metallicity, init_mass_lv, filename in tqdm(
        sample_records,
        total=len(sample_records),
    )
}


In [ ]:
sample_record_df

In [ ]:
plot_color_dict = {
    1: "#711415",
    2: "#ae311e",
    3: "#f37324",
    4: "#f6a020",
    5: "#f8cc1b",
    6: "#b5be2f",
    7: "#72b043",
    8: "#007f4e",
}

fig, axes = plt.subplots(
    2, 3, figsize=(17, 11), dpi=300, gridspec_kw={"wspace": 0.3, "hspace": 0.4}
)

attr_pairs = [(2, 4), (2, 8), (6, 8), (14, 4), (14, 8), (14, 12)]

for ax_label, (ax, (init_metallicity, init_gc_radius)) in enumerate(
    zip(axes.flatten(), attr_pairs)
):
    ax.set_title(
        rf"$Z_{{init.}}={init_metallicity * 10e-4},\;"
        rf"R_\mathrm{{GC,\,init.}}={init_gc_radius}\ \mathrm{{kpc}}$",
        fontsize=20,
        y=1.02,
    )

    for init_mass_lv in INIT_MASS_LV_TO_PLOT:
        ax.plot(
            *sample_data_dict[(init_gc_radius, init_metallicity, init_mass_lv)].T,
            ls="",
            marker="o",
            mew=0,
            markersize=2.2 + 0.1 * init_mass_lv,
            color=plot_color_dict[init_mass_lv],
            label=rf"$M_{{init.}}={10**init_mass_lv}\ M_\odot$",
            alpha=0.15 + 0.1 * init_mass_lv,
            zorder=init_mass_lv,
        )

    ax.text(
        0.95,
        0.95,
        f"$({ax_label + 1})$",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=22,
    )

    ax.set_aspect("equal")
    ax.grid(ls=":", lw=0.8, c="darkgrey", zorder=-1)
    ax.set_xlim(-1.4, 1.4)
    ax.set_xlabel(r"$X$ [$\mathrm{pc}$]", fontsize=20)
    ax.xaxis.set_major_locator(plt.MultipleLocator(0.8))
    ax.xaxis.set_minor_locator(plt.MultipleLocator(0.2))
    ax.set_ylim(-1.4, 1.4)
    ax.set_ylabel(r"$Y$ [$\mathrm{pc}$]", fontsize=20)
    ax.yaxis.set_major_locator(plt.MultipleLocator(0.8))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.2))

# legend
legend_handles = [
    mpl.lines.Line2D(
        [0],
        [0],
        ls="",
        marker="o",
        markersize=6,
        color=plot_color_dict[init_mass_lv],
        label=f"M{init_mass_lv}",
        alpha=0.8,
    )
    for init_mass_lv in INIT_MASS_LV_TO_PLOT
]
fig.legend(
    handles=legend_handles,
    title="Model Family",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncols=len(INIT_MASS_LV_TO_PLOT),
    frameon=True,
    fontsize=18,
    title_fontsize=20,
)

fig_export_path = OUTPUT_BASE / "figures" / "overall"
fig_export_path.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_export_path / "sample_t0_xy_distribution.pdf")